# Ouroboros — Colab setup (A100)

Run top to bottom on a GPU runtime. Mounts Drive, clones the repo, installs pinned deps (keeping Colab's CUDA PyTorch), copies WebDataset shards to local disk, and runs the test suite. All project code runs in subprocesses (`!python ...`) so that pinned numpy etc. take effect without restarting the kernel.

If the repo is private, add a Colab secret `GITHUB_TOKEN` (key icon in the left sidebar) with read access to the repository.

In [ ]:
import time, os, subprocess
T0 = time.time()
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python --version

In [ ]:
# ---- configuration ----
REPO = 'yaniguan/ouroboros-ocsr'
BRANCH = 'claude/vigilant-johnson-j4882f'  # set to 'main' once merged
DRIVE_ROOT = '/content/drive/MyDrive/ouroboros'  # checkpoints, shards, results live here
LOCAL_SHARDS = '/content/shards'
REPO_DIR = '/content/ouroboros-ocsr'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_ROOT, exist_ok=True)
for sub in ('shards', 'runs', 'results'):
    os.makedirs(f'{DRIVE_ROOT}/{sub}', exist_ok=True)

In [ ]:
token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-b', BRANCH, url, REPO_DIR], check=True)
!git -C {REPO_DIR} log --oneline -1

In [ ]:
%%bash -s "$REPO_DIR"
set -e
cd "$1"
# py3nj (escnn dependency) builds from source and needs a Fortran compiler
which gfortran || (apt-get -qq update && apt-get -qq install -y gfortran > /dev/null)
pip install -q -r requirements-colab.txt
pip install -q --no-deps -e .
python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# Copy shards Drive -> local disk (Drive FUSE is too slow for training I/O).
src = f'{DRIVE_ROOT}/shards'
os.makedirs(LOCAL_SHARDS, exist_ok=True)
if any(True for _ in os.scandir(src)):
    !rsync -a --info=progress2 {src}/ {LOCAL_SHARDS}/
else:
    print('no shards on Drive yet; skipping copy')
!du -sh {LOCAL_SHARDS}

In [ ]:
!cd {REPO_DIR} && python -c "import escnn, mace; from escnn import gspaces; from mace.calculators import mace_off; import rdkit; print('escnn/mace/rdkit import OK', rdkit.__version__)"

In [ ]:
!cd {REPO_DIR} && python -m pytest -q

In [ ]:
print(f'wall time: {(time.time() - T0) / 60:.1f} min')
!pip freeze | grep -iE '^(torch|escnn|mace-torch|rdkit|numpy|e3nn)=='